# Project: Teaching an LLM to Reason

Select **Jupyter Kernel > Python (venv2)** before running. This completed notebook fills every TODO for LoRA setup, prompt engineering, reward shaping, GRPO training, and evaluation.

In [ ]:
!pip install -q ipython-autotime
%load_ext autotime

In [ ]:
!nvidia-smi

In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch, re, random

max_seq_length = 384
# Rank 16 balances adapter capacity and T4 memory for this narrow task.
lora_rank = 16
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct", max_seq_length=max_seq_length,
    load_in_4bit=True, fast_inference=True, max_lora_rank=lora_rank,
    gpu_memory_utilization=0.5,
)
model = FastLanguageModel.get_peft_model(
    model, r=lora_rank,
    # Adapt all attention and MLP projections for sufficient task-learning capacity.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_rank, use_gradient_checkpointing="unsloth",
)

## Prompt engineering

In [ ]:
from vllm import SamplingParams
SYSTEM_PROMPT = ""
USER_PROMPT = 'How many of the letter "g" are there in the word "engage"'
text_for_completion = tokenizer.apply_chat_template([
    {"role":"system","content":SYSTEM_PROMPT}, {"role":"user","content":USER_PROMPT}],
    tokenize=False, add_generation_prompt=True)
sampling_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=2048)
output = model.fast_generate([text_for_completion], sampling_params=sampling_params, lora_request=None)[0].outputs[0].text
print("=== TEXT FOR COMPLETION ===")
print(text_for_completion)
print("=== GENERATED OUTPUT ===")
print(output)

In [ ]:
SYSTEM_PROMPT = """
Count the requested letter by inspecting the word one character at a time. Keep a running total. Do not skip or add characters. Return only this structure:
<reasoning>
Letter-by-letter spelling:
1. [first character] - [running count] so far
2. [second character] - [running count] so far
...
</reasoning>
<answer>
[final integer]
</answer>

Example:
User: How many of the letter "o" are there in the word "room"
Assistant:
<reasoning>
Letter-by-letter spelling:
1. r - 0 so far
2. o - 1 so far
3. o - 2 so far
4. m - 2 so far
</reasoning>
<answer>
2
</answer>
"""
USER_PROMPT = 'How many of the letter "g" are there in the word "engage"'
text_for_completion = tokenizer.apply_chat_template([
    {"role":"system","content":SYSTEM_PROMPT}, {"role":"user","content":USER_PROMPT}],
    tokenize=False, add_generation_prompt=True)
output = model.fast_generate([text_for_completion], sampling_params=sampling_params, lora_request=None)[0].outputs[0].text
print("=== TEXT FOR COMPLETION ===")
print(text_for_completion)
print("=== GENERATED OUTPUT ===")
print(output)

## Dataset

In [ ]:
ALL_WORDS = [
"idea","glow","rust","maze","echo","wisp","veto","lush","gaze","knit","fume","plow","void","oath","grim","crisp","lunar","fable","quest","verge","brawn","elude","aisle","ember","crave","ivory","mirth","knack","wryly","onset","mosaic","velvet","sphinx","radius","summit","banner","cipher","glisten","mantle","scarab","expose","fathom","tavern","fusion","relish","lantern","enchant","torrent","capture","orchard","eclipse","frescos","triumph","absolve","gossipy","prelude","whistle","resolve","zealous","mirage","aperture","sapphire"]
print(len(ALL_WORDS)); ALL_WORDS[:10]

In [ ]:
from datasets import Dataset

def generate_records():
    for word in ALL_WORDS:
        for letter in sorted(set(word)):
            yield {"words":word, "letters":letter, "counts":word.count(letter)}
        left = len(word)//7 + 1
        random.seed(word)
        alphabet=list("abcdefghijklmnopqrstuvwxyz"); random.shuffle(alphabet)
        for letter in alphabet:
            if letter not in word:
                yield {"words":word, "letters":letter, "counts":0}; left -= 1
            if left == 0: break

ds=Dataset.from_generator(generate_records)
ds[0]

In [ ]:
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
Counting the number of [letter_to_count]'s in the word [word]
1. [first letter] - [count of requested letter so far] so far
2. [second letter] - [count of requested letter so far] so far
...
</reasoning>
<answer>
[number]
</answer>
"""
ds=ds.map(lambda x:{"prompt":[
    {"role":"system","content":SYSTEM_PROMPT},
    {"role":"user","content":'How many of the letter "{}" are there in the word "{}"'.format(x["letters"],x["words"])}]})
ds[0]

In [ ]:
text=tokenizer.apply_chat_template(ds[0]["prompt"],tokenize=False,add_generation_prompt=True)
output=model.fast_generate([text],sampling_params=SamplingParams(temperature=0.8,top_p=0.95,max_tokens=1024),lora_request=None)[0].outputs[0].text
print(output)

## Reward functions

In [ ]:
def extract_letter_numbering(response):
    matches=re.findall(r"(?:^|\n)\s*(\d+)\.\s+[a-z]",response,flags=re.I)
    return [int(m) for m in matches]

assert extract_letter_numbering("\n1. g - 1 so far\n2. o - 1 so far\n3. a - 2 so far\n4. a - 2 so far\n5. l - 2 so far") == [1,2,3,4,5]

def numbering_reward_func(completions,words,**kwargs)->list[float]:
    responses=[c[0]["content"] for c in completions]; result=[]
    for response,word in zip(responses,words):
        reward=0.0
        for ix,n in enumerate(extract_letter_numbering(response)):
            line=ix+1
            reward += 1.0 if n==line else -1.0
            if line>len(word): reward -= 1.0
        result.append(reward/max(len(word),1))
    return result

_bad="<reasoning>\n1. g - 1 so far\n2. o - 1 so far\n3. a - 2 so far\n3. l - 2 so far\n1. l - 2 so far\n1. l - 2 so far\n</reasoning><answer>2</answer>"
_better="<reasoning>\n1. g - 1 so far\n2. o - 1 so far\n3. a - 2 so far\n3. l - 2 so far\n</reasoning><answer>2</answer>"
res=numbering_reward_func([[{"content":_bad}],[{"content":_better}]],words=["goal","goal"])
print(res); assert res[1]>res[0]

In [ ]:
def extract_spelling(response):
    return "".join(re.findall(r"(?:^|\n)\s*\d+\.\s+([a-z])",response,flags=re.I)).lower()

assert extract_spelling("\n1. g - 1 so far\n2. o - 1 so far\n3. a - 2 so far\n3. l - 2 so far\n5. l - 2 so far") == "goall"

def spelling_reward_func(completions,words,**kwargs)->list[float]:
    from collections import Counter
    result=[]
    for word,completion in zip(words,completions):
        spelling=extract_spelling(completion[0]["content"]); target=word.lower()
        reward=2.0 if spelling==target else 0.0
        reward -= 0.5*abs(len(spelling)-len(target))
        reward -= 0.5*sum((Counter(spelling)-Counter(target)).values())
        reward -= 0.5*sum((Counter(target)-Counter(spelling)).values())
        result.append(reward)
    return result

res=spelling_reward_func([[{"content":"\n1. g - 1 so far\n2. o - 1 so far\n3. a - 1 so far\n4. l - 1 so far\n5. l - 1 so far"}],[{"content":"\n1. g - 1 so far\n2. o - 1 so far\n3. a - 1 so far\n4. l - 1 so far"}]],words=["goal","goal"])
print(res); assert res[1]>res[0]

In [ ]:
def get_resp_letters_and_counts(response):
    return [(letter.lower(),count) for letter,count in re.findall(r"(?:^|\n)\s*\d+\.\s+([a-z])\D*(\d+)",response,flags=re.I)]

assert get_resp_letters_and_counts("\n1. g - 1 so far\n2. o - 1 so far\n3. a - 2 so far\n4. a - 2 so far\n5. l - 2 so far") == [("g","1"),("o","1"),("a","2"),("a","2"),("l","2")]

def counting_reward_func(completions,letters,**kwargs)->list[float]:
    result=[]
    for letter,completion in zip(letters,completions):
        pairs=get_resp_letters_and_counts(completion[0]["content"])
        if not pairs: result.append(-1.0); continue
        actual=0; reward=0.0
        for resp_letter,resp_count in pairs:
            if letter.lower()==resp_letter: actual+=1
            reward += 1.0 if int(resp_count)==actual else -1.0
        result.append(reward/len(pairs))
    return result

worse="\n1. g - 0 so far\n2. o - 0 so far\n3. a - 1 so far\n4. a - 2 so far\n5. l - 0 so far"
better="\n1. g - 1 so far\n2. o - 1 so far\n3. a - 1 so far\n4. a - 1 so far\n5. l - 1 so far"
res=counting_reward_func([[{"content":worse}],[{"content":better}]],letters=["g","g"])
print(res); assert res[1]>res[0]

In [ ]:
def extract_xml_answer(text:str)->str:
    match=re.search(r"<answer>(.*?)</answer>",text,re.S|re.I)
    return match.group(1).strip() if match else ""

def format_reward_func(completions,**kwargs)->list[float]:
    pattern=r"^\s*<reasoning>.*?</reasoning>\s*<answer>.*?</answer>\s*$"
    result=[]
    for completion in completions:
        response=completion[0]["content"]; reward=0.0
        if re.match(pattern,response,re.S|re.I): reward+=0.5
        if re.fullmatch(r"-?\d+",extract_xml_answer(response)): reward+=0.5
        result.append(reward)
    return result

res=format_reward_func([[{"content":"This is my answer"}],[{"content":"<reasoning>Reason.</reasoning><answer>3</answer>"}]])
print(res); assert res[1]>res[0]

In [ ]:
def correct_answer_reward_func(prompts,completions,counts,**kwargs)->list[float]:
    responses=[c[0]["content"] for c in completions]
    extracted=[extract_xml_answer(r) for r in responses]
    print(f"Question: {prompts[0][-1]['content']}\nAnswer: {counts[0]}\nResponse: {responses[0]}\nExtracted: {extracted[0]}")
    return [2.0 if str(r)==str(a) else 0.0 for r,a in zip(extracted,counts)]

res=correct_answer_reward_func(
 prompts=[[{"content":"How many..."}],[{"content":"How many..."}]],
 completions=[[{"content":"<reasoning>...</reasoning><answer>3</answer>"}],[{"content":"<reasoning>...</reasoning><answer>3</answer>"}]],counts=[0,3])
print(res); assert res[1]>res[0]

REWARD_FUNCS=[numbering_reward_func,spelling_reward_func,counting_reward_func,format_reward_func,correct_answer_reward_func]

## Train

In [ ]:
COMMON_GRPO_TRAINING_PARAMS=dict(
 learning_rate=5e-6, beta=0.04,
 per_device_train_batch_size=8, num_generations=8, gradient_accumulation_steps=1,
 adam_beta1=0.9, adam_beta2=0.99, weight_decay=0.1, warmup_ratio=0.1,
 lr_scheduler_type="cosine", optim="adamw_8bit", logging_steps=1,
 max_prompt_length=256, max_completion_length=200, num_train_epochs=1,
 save_steps=250, max_grad_norm=0.1, report_to="none", output_dir="outputs", use_vllm=True)

In [ ]:
from trl import GRPOConfig, GRPOTrainer
training_args=GRPOConfig(**COMMON_GRPO_TRAINING_PARAMS,max_steps=5)
trainer=GRPOTrainer(model=model,processing_class=tokenizer,reward_funcs=REWARD_FUNCS,args=training_args,train_dataset=ds)
trainer_res=trainer.train()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
print(f"available columns: {trainer.state.log_history[0].keys()}")
log_df=pd.DataFrame(trainer.state.log_history)
log_df["reward"].plot(); log_df["rewards/correct_answer_reward_func/mean"].plot()
plt.legend(["reward","rewards/correct_answer_reward_func/mean"]); plt.show()

In [ ]:
# Medium full run. The quick run above validates memory and reward behavior first.
training_args=GRPOConfig(**COMMON_GRPO_TRAINING_PARAMS,max_steps=100)
trainer=GRPOTrainer(model=model,processing_class=tokenizer,reward_funcs=REWARD_FUNCS,args=training_args,train_dataset=ds)
trainer_res=trainer.train()

In [ ]:
log_df=pd.DataFrame(trainer.state.log_history)
log_df["reward"].plot(); log_df["rewards/correct_answer_reward_func/mean"].plot()
plt.legend(["reward","rewards/correct_answer_reward_func/mean"]); plt.show()

## View results

In [ ]:
model.save_lora("grpo_saved_lora")

def compare_old_and_new_model(messages):
    text=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    params=SamplingParams(temperature=0.8,top_p=0.95,max_tokens=1024)
    old=model.fast_generate(text,sampling_params=params)[0].outputs[0].text
    new=model.fast_generate(text,sampling_params=params,lora_request=model.load_lora("grpo_saved_lora"))[0].outputs[0].text
    print("===OLD===\n",old,"\n\n===NEW===\n",new)

In [ ]:
# First dataset item
compare_old_and_new_model(ds[0]["prompt"])

In [ ]:
# Retention check
compare_old_and_new_model([
 {"role":"system","content":"Answer briefly and accurately."},
 {"role":"user","content":"What is the capital city of France?"},
])

## Completed choices

- `lora_rank = 16`
- All attention and MLP projection adapters selected
- Five reward functions completed and unit-tested
- GRPO uses learning rate `5e-6`, beta `0.04`, batch size `8`, eight generations, and one accumulation step
- Full run uses `max_steps=100`
- Dataset and general-knowledge comparison cells completed

Training is stochastic. If GPU memory is insufficient, lower the batch/generation grouping consistently and rerun the five-step validation.